# 🚀 Distributed Semiconductor Image Restoration on Kaggle (T4 x2 Dual-GPU)

This notebook trains **NAFNet-SR** using **Dual NVIDIA Tesla T4 GPUs (T4 x2)** with **PyTorch DataParallel**, **Automatic Mixed Precision (AMP FP16)**, **Model EMA**, and **Calibrated Composite Metrology Loss**.


## Step 1: Verify Dual GPU (Tesla T4 x2)


In [ ]:
!nvidia-smi
import torch
print('CUDA Available:', torch.cuda.is_available())
print('GPU Count:', torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f'  [+] GPU {i}: {torch.cuda.get_device_name(i)}')


## Step 2: Clone Repository (Branch `Kunal`)


In [ ]:
!git clone -b Kunal https://github.com/kmbeddedd/semicon_2026.git
%cd semicon_2026


## Step 3: Install Dependencies


In [ ]:
!pip install -q -r requirements.txt


## Step 4: Auto-Detect Dataset from `/kaggle/input/` & Create Validation Split

This cell checks `/kaggle/input/` and sets up `data/train/NoisyLR`, `data/train/GT`, `data/val/NoisyLR`, and `data/val/GT`.

In [ ]:
import os, glob, shutil, random, zipfile

print('=== Step 4: Inspecting /kaggle/input/ ===')
all_input_files = glob.glob('/kaggle/input/**/*', recursive=True)
print(f'Total items found in /kaggle/input: {len(all_input_files)}')
for p in all_input_files[:15]:
    print('  ->', p)
if len(all_input_files) > 15:
    print(f'  ... and {len(all_input_files) - 15} more files')

os.makedirs('data/train', exist_ok=True)

# 1. If any zip archives are in /kaggle/input, extract them
for z in glob.glob('/kaggle/input/**/*.zip', recursive=True):
    print(f'[+] Extracting zip archive: {z} -> data/...')
    try:
        with zipfile.ZipFile(z, 'r') as zip_ref:
            zip_ref.extractall('data')
        print(f'[+] Successfully extracted {z}')
    except Exception as e:
        print(f'[!] Zip extraction error: {e}')

# 2. Find and link NoisyLR and GT folders from /kaggle/input if unzipped
if not os.path.exists('data/train/NoisyLR') or len(os.listdir('data/train/NoisyLR')) == 0:
    for root, dirs, _ in os.walk('/kaggle/input'):
        for d in dirs:
            d_lower = d.lower()
            full_dir = os.path.join(root, d)
            if 'noisylr' in d_lower or 'noisy_lr' in d_lower:
                print(f'[+] Found NoisyLR directory: {full_dir}')
                shutil.copytree(full_dir, 'data/train/NoisyLR', dirs_exist_ok=True)
            elif d == 'GT' or d_lower == 'gt' or 'groundtruth' in d_lower:
                print(f'[+] Found GT directory: {full_dir}')
                shutil.copytree(full_dir, 'data/train/GT', dirs_exist_ok=True)

# 3. Gather all training images
exts = ('*.npy', '*.NPY', '*.png', '*.PNG', '*.jpg', '*.JPG', '*.jpeg', '*.JPEG', '*.tif', '*.TIF')
train_files = []
if os.path.exists('data/train/NoisyLR'):
    for ext in exts:
        train_files.extend(glob.glob(f'data/train/NoisyLR/{ext}'))
train_files = sorted(list(set(train_files)))

print(f'\n[+] Detected {len(train_files)} training samples in data/train/NoisyLR')

# 4. Safe Validation Split Creation (Guarded against empty lists)
if len(train_files) > 0 and not os.path.exists('data/val'):
    random.seed(42)
    os.makedirs('data/val/NoisyLR', exist_ok=True)
    os.makedirs('data/val/GT', exist_ok=True)
    val_k = max(1, int(len(train_files) * 0.1))
    val_k = min(val_k, len(train_files))
    val_samples = random.sample(train_files, k=val_k)
    for vf in val_samples:
        fname = os.path.basename(vf)
        shutil.copy(vf, os.path.join('data/val/NoisyLR', fname))
        gt_path = os.path.join('data/train/GT', fname)
        if os.path.exists(gt_path):
            shutil.copy(gt_path, os.path.join('data/val/GT', fname))
    print(f'[+] Successfully created 10% validation split: {len(val_samples)} samples in data/val/')
elif len(train_files) == 0:
    print('\n⚠️ WARNING: No training images were found in data/train/NoisyLR!')
    print('👉 Action Required: Click "+ Add Input" on the right panel in Kaggle and attach your dataset or train.zip.')

print('\n=== Dataset Setup Summary ===')
print('Train NoisyLR count:', len(glob.glob('data/train/NoisyLR/*.*')) if os.path.exists('data/train/NoisyLR') else 0)
print('Train GT count:     ', len(glob.glob('data/train/GT/*.*')) if os.path.exists('data/train/GT') else 0)
print('Val NoisyLR count:  ', len(glob.glob('data/val/NoisyLR/*.*')) if os.path.exists('data/val/NoisyLR') else 0)
print('Val GT count:       ', len(glob.glob('data/val/GT/*.*')) if os.path.exists('data/val/GT') else 0)


## Step 5: Train NAFNet-SR on Dual T4 GPUs (Batch Size 64: 32 per GPU)


In [ ]:
!python train.py --epochs 100 --batch_size 64 --lr 8e-4 --warmup_epochs 5 --scale 2 --patch_size 0 --num_workers 4 --save_dir /kaggle/working/weights


## Step 6: Evaluate & Benchmark (Single Pass & 8-Fold TTA)


In [ ]:
# Fast production evaluation (< 14ms)
!python eval.py --input_dir data/val/NoisyLR --target_dir data/val/GT --output_dir /kaggle/working/val_restored --weights /kaggle/working/weights/best_model.pt --scale 2 --batch_size 16 --no_tta --check_clean_damage

# 8-Fold TTA evaluation
!python eval.py --input_dir data/val/NoisyLR --target_dir data/val/GT --output_dir /kaggle/working/val_restored_tta --weights /kaggle/working/weights/best_model.pt --scale 2 --batch_size 16
